# Quantum-limited imaging using diffractive optical neural networks — Fig. 2

Companion notebook for

> A. Warke, A. Zhang, A. I. Lvovsky, *Quantum-limited imaging using diffractive optical neural networks*, [arXiv:2608.12300](https://arxiv.org/abs/2608.12300).

**Fig. 2 — two-parameter estimation.** Low-contrast object $f(x) = a_0 + a_1\cos(k_1 x) + a_2\cos(k_2 x)$, $k_m = \pi m/L$ (DCT-II), with $k_1$ fixed ($m_1 = 20$) and $k_2$ swept across the band. All quantities are per-photon values of $\mathrm{Tr\,Cov}(a_1, a_2)$:

- **bound hierarchy** $c_Q \le c_{\rm NH} \le c_{\rm ansatz}$ — QCRB, NHCRB from the Nagaoka–Hayashi SDP, and the SLD-ansatz upper bound; the shaded band between QCRB and ansatz is where the true Nagaoka bound must lie. In contrast to Fig. 1, the NHCRB **detaches** from the QCRB;
- **direct imaging** — numerical CRB, analytic reference, and Monte-Carlo variance;
- **DONN** — CRB of the trained cascade and its Monte-Carlo variance, tracking the NHCRB rather than the (unattainable) QCRB.

**Precisions:** the DONN mask search runs in `float32`/`complex64` (large GPU speed-up); every reported bound and the Monte Carlo are evaluated in NumPy/`complex128` float64 — only the mask *search* is approximate, the reported numbers are not.

**Outputs:** `figures/WZL_Fig2.png` / `.svg` and `data/WZL_Fig2.npz` (plus a per-point checkpoint `data/WZL_Fig2_ckpt.npz`).

**Requirements:** `numpy`, `torch`, `matplotlib`, `cvxpy` (Clarabel). A CUDA GPU is used automatically if available. The cell marked **[slow]** retrains a DONN and solves one SDP per sweep point (15 points); the checkpoint file is updated after every point, so a partial run can already be re-plotted. To regenerate the figure from saved data only, run all cells *except* the **[slow]** one.

In [ ]:
%matplotlib inline
import time
import numpy as np, cvxpy as cp, torch, torch.nn as nn, torch.optim as optim
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------ device / precision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RDTYPE = torch.float32        # real dtype    -> torch.float32 for a large GPU speed-up
CDTYPE = torch.complex64      # complex dtype -> must match RDTYPE

torch.set_default_dtype(RDTYPE)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False   # keep full precision if RDTYPE=float32
    print(f"training on {torch.cuda.get_device_name(0)}  ({RDTYPE})", flush=True)
else:
    print("CUDA not available - training on CPU", flush=True)

torch.manual_seed(0); np.random.seed(0)

# ------------------------------------------------------------------ knobs
NA_D, LAM_D, L_D, N_D = 1.4, 0.540, 10.8, 256
LADDER   = (2, 4, 8, 10)      # warm-start plane ladder
EPOCHS   = 1500               # Adam steps per ladder stage
LR0      = 0.04
LR_DECAY = 0.5
MC_RUNS  = 5000
MC_NPH   = 100000
SDP_TRUNC_TOL = 1e-4          # support truncation before the NHCRB SDP

## Optics: 1D incoherent imaging model

Identical to Fig. 1: exact band-limited reproducing kernel (sinc Gram matrix, factorised as $G = TT^\top$), detector-plane amplitudes $\psi_j(x)$ on a grid of span $2L$, Fourier-cosine object parametrisation, triangle OTF, and projection of $\rho,\ \partial_a\rho$ onto the truncated eigenbasis.

In [ ]:
def optics_exact_1d(NA=NA_D, lam=LAM_D, L=L_D, N=N_D, det_span=2, tol=1e-12,
                    normalize_det=False):
    dx = L/N
    nc = NA/lam
    x_mid = (np.arange(N)+0.5)*dx

    # exact band-limited Gram -- grid-free, no padding, no wraparound
    G = np.sinc(2*nc*(x_mid[:,None]-x_mid[None,:]))
    ev, V = np.linalg.eigh(G); keep = ev > tol*ev.max()
    T = V[:,keep]*np.sqrt(ev[keep])                   # (N,K): rho = T.T@(w[:,None]*T)

    # detector / propagation grid, centred on the object, det_span*L wide
    Nd = det_span*N
    x_det = (np.arange(Nd)+0.5)*dx - (Nd-N)//2*dx
    psi  = np.sqrt(2*nc)*np.sinc(2*nc*(x_det[None,:]-x_mid[:,None]))*np.sqrt(dx)
    P_wf = psi**2
    capture = P_wf.sum(1)
    if normalize_det:
        psi = psi/np.sqrt(capture)[:,None]; P_wf = psi**2

    return dict(x_mid=x_mid, x_det=x_det, T=T, K=int(keep.sum()), N=N, Nd=Nd,
                L=L, dx=dx, nu_coh=nc, nu_incoh=2*nc, capture=capture,
                M_max=int(np.floor(4*NA*L/lam)), psi=psi, P_wf=P_wf)

def cos_derivs(m_list, opt, a0=1.0):
    L, N, dx, x = opt['L'], opt['N'], opt['dx'], opt['x_mid']
    ft = a0*N*dx; w = np.full(N, a0)/ft*dx
    dw = np.zeros((len(m_list), N))
    for i, m in enumerate(m_list):
        C = np.cos(np.pi*m/L*x); dft = np.sum(C)*dx
        dw[i] = (C*ft - a0*dft)/ft**2*dx
    return w, dw

def otf_triangle(nu, opt):                    # exact incoherent OTF, hard 1D pupil
    return np.clip(1.0 - nu/opt['nu_incoh'], 0.0, 1.0)

def eigenbasis(w, dw_list, opt, tol=1e-10):
    T = opt['T']
    rho = T.T @ (w[:,None]*T)
    drf = [T.T @ (dw[:,None]*T) for dw in dw_list]
    ev, U = np.linalg.eigh(rho); keep = ev > tol*ev.max()
    lamk, Uk = ev[keep], U[:,keep]
    return np.diag(lamk), [Uk.T@dr@Uk for dr in drf], len(lamk)

## Precision bounds

- `crb_di` — classical CRB of ideal photon counting (direct imaging).
- `qfi_matrix` / `crb_qfi` / `slds` — SLD quantum Fisher information and SLDs for a general $\rho_0$ (not assuming $\rho_0 = I/K$).
- `ansatz_ub` — Nagaoka ($n=2$) objective evaluated at the feasible point $X_j = \sum_k [Q^{-1}]_{kj} L_k$: an upper bound on $c_{\rm NH}$ for any $\rho_0$, since feasibility is all that is required. Includes local-unbiasedness self-checks.
- `analytic_ub` — closed form from the $\rho_0 = I/K$ derivation, kept as a reference only (with the exact optics $\rho_0$ has a Slepian spectrum, so it is no longer derived).
- `crb_nhcrb` — Nagaoka–Hayashi SDP (Clarabel). The block side is $(M{+}1)K$ and memory grows steeply in $K$, so the support of $\rho_0$ is truncated first (`truncate_support`); the bound is insensitive to the truncation tolerance.

In [ ]:
def crb_di(w, dw_list, opt, Nph=1.0):
    P = opt['P_wf']; M = len(dw_list)
    u = Nph*(w[:,None]*P).sum(0); du = [Nph*(dw[:,None]*P).sum(0) for dw in dw_list]
    F = np.array([[np.sum(du[a]*du[b]/(u+1e-15)) for b in range(M)] for a in range(M)])
    return np.trace(np.linalg.inv(F))

def qfi_matrix(rho_eig, dr_list, Nph=1.0):
    lam = np.diag(rho_eig); M = len(dr_list)
    den = lam[:,None]+lam[None,:]; ok = den > 1e-12*lam.max()
    inv = np.zeros_like(den); inv[ok] = 1.0/den[ok]
    return np.array([[Nph*2*np.sum(dr_list[a]*dr_list[b]*inv).real
                      for b in range(M)] for a in range(M)])

def crb_qfi(rho_eig, dr_list, Nph=1.0):
    return np.trace(np.linalg.inv(qfi_matrix(rho_eig, dr_list, Nph)))

def slds(rho_eig, dr_list):
    lam = np.diag(rho_eig)
    den = lam[:,None] + lam[None,:]
    ok = den > 1e-12*lam.max()
    inv = np.zeros_like(den); inv[ok] = 1.0/den[ok]
    return [2*dr*inv for dr in dr_list]

def ansatz_ub(rho_eig, dr_list, check=True):
    assert len(dr_list) == 2, "Nagaoka n=2 form: exactly two parameters"
    Ls = slds(rho_eig, dr_list)
    Q = qfi_matrix(rho_eig, dr_list); Qinv = np.linalg.inv(Q)
    X = [sum(Qinv[k,j]*Ls[k] for k in range(2)) for j in range(2)]
    term1 = sum(np.trace(rho_eig @ (X[j] @ X[j])) for j in range(2)).real
    C = rho_eig @ (X[0]@X[1] - X[1]@X[0])          # anti-Hermitian
    term2 = float(np.sum(np.abs(np.linalg.eigvals(C))))
    if check:
        U = np.array([[np.trace(dr_list[i] @ X[j]) for j in range(2)] for i in range(2)])
        assert np.allclose(U, np.eye(2), atol=1e-8), "ansatz not locally unbiased"
        assert abs(term1 - np.trace(Qinv)) < 1e-8*max(1, abs(term1)), "term1 != Tr(Q^-1)"
    return float(term1 + term2), float(np.trace(Qinv))

def analytic_ub(nu1, nu2, opt, a0=1.0):
    o1, o2 = otf_triangle(nu1, opt), otf_triangle(nu2, opt)
    if o1 <= 0 or o2 <= 0: return np.inf
    wmin = min(o1, o2, 1-o1, 1-o2)
    return 2*a0**2*(1/o1 + 1/o2 + 2*wmin/(o1*o2))

def truncate_support(rho_eig, dr_list, tol=SDP_TRUNC_TOL):
    ev, U = np.linalg.eigh(rho_eig)
    Uk = U[:, ev > tol*ev.max()]
    return Uk.T @ rho_eig @ Uk, [Uk.T @ d @ Uk for d in dr_list], Uk.shape[1]

def crb_nhcrb(rho_eig, dr_list, Nph=1.0, trunc_tol=SDP_TRUNC_TOL,
              solver=cp.CLARABEL, max_iters=20000, eps=1e-4, verbose=False):
    rho_eig, dr_list, K = truncate_support(rho_eig, dr_list, trunc_tol)
    M = len(dr_list)
    if K < 2: return np.inf
    dn = np.array([np.linalg.norm(dd) for dd in dr_list])
    drt = [dd/nn for dd, nn in zip(dr_list, dn)]
    wgt = 1.0/dn**2
    Xs = [cp.Variable((K, K), symmetric=True) for _ in range(M)]
    Ls = {(j, k): cp.Variable((K, K), symmetric=True) for j in range(M) for k in range(j, M)}
    gL = lambda j, k: Ls[(j, k)] if j <= k else Ls[(k, j)].T
    cons = [cp.trace(drt[j]@Xs[k]) == (1 if j == k else 0) for j in range(M) for k in range(M)]
    cons.append(cp.bmat([[gL(j, k) for k in range(M)]+[Xs[j]] for j in range(M)]
                        + [[Xs[m].T for m in range(M)]+[np.eye(K)]]) >> 0)
    prob = cp.Problem(cp.Minimize(sum(wgt[j]*cp.trace(rho_eig@Ls[(j, j)]) for j in range(M))), cons)
    try:
        kw = {} if solver == cp.CLARABEL else dict(max_iters=max_iters, eps=eps)
        prob.solve(solver=solver, verbose=verbose, **kw)
        if prob.value is not None and np.isfinite(prob.value): return prob.value/Nph
    except Exception as e:
        print(f"      SDP failed: {type(e).__name__}: {str(e)[:100]}", flush=True)
    return np.inf

## DONN: simultaneous two-parameter estimation

`MPLC` propagates the per-emitter fields through `n_planes` trainable phase masks separated by optical Fourier transforms and minimises the joint $\mathrm{Tr}\,F^{-1}$ (2×2 Fisher inverse). `donn_planes` trains over the warm-start ladder 2 → 4 → 8 → 10 with cosine-annealed Adam; each stage seeds the next with near-identity extra planes, which beats a fresh deep model at equal epoch budget (especially near cutoff). `donn_detector` recomputes the trained detector intensities in `complex128` on the CPU, so the Monte-Carlo arm is not limited by the training dtype.

In [ ]:
class MPLC(nn.Module):

    def __init__(self, opt, m_list, n_planes, a0=1.0, init=None, gen=None, device=DEVICE):
        super().__init__()
        Nd = opt['psi'].shape[1]
        self.n = n_planes; self.M = len(m_list)
        self.register_buffer('psi', torch.tensor(opt['psi'], dtype=CDTYPE, device=device))
        w, dw = cos_derivs(m_list, opt, a0)
        self.register_buffer('w',  torch.tensor(w,  dtype=RDTYPE, device=device))
        self.register_buffer('dw', torch.tensor(dw, dtype=RDTYPE, device=device))
        if init is None:
            init = [torch.randn(Nd, generator=gen, dtype=RDTYPE)*0.5 for _ in range(n_planes)]
        self.masks = nn.ParameterList([
            nn.Parameter(p.detach().clone().to(device=device, dtype=RDTYPE)) for p in init])

    def _U(self, psi):
        for k in range(self.n):
            psi = psi*torch.exp(1j*self.masks[k].to(CDTYPE)).unsqueeze(0)
            psi = torch.fft.fftshift(torch.fft.fft(torch.fft.ifftshift(psi, dim=1),
                                                   norm='ortho'), dim=1)
        return psi

    def forward(self):
        Pd = torch.abs(self._U(self.psi))**2
        u = torch.clamp((self.w.unsqueeze(1)*Pd).sum(0), min=1e-12)
        du = torch.einsum('mj,jd->md', self.dw, Pd)
        eye = torch.eye(self.M, dtype=Pd.dtype, device=Pd.device)   # device-aware!
        F = (du/u) @ du.T + 1e-12*eye
        return torch.trace(torch.linalg.inv(F))

def _sync(device=DEVICE):
    if device.type == "cuda":
        torch.cuda.synchronize()

def _train(model, epochs, lr, cosine=True):
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None
    best = np.inf; bm = None
    for _ in range(epochs):
        opt.zero_grad(); v = model(); v.backward(); opt.step()
        if sched is not None: sched.step()
        vi = v.item()
        if vi < best: best = vi; bm = [p.detach().clone() for p in model.masks]
    return best, bm

def donn_planes(opt, m_list, ladder=LADDER, epochs=EPOCHS, lr=LR0,
                lr_decay=LR_DECAY, jitter=1e-3, seed=0, device=DEVICE, verbose=False):
    Nd = opt['psi'].shape[1]
    best_val, best_masks, prev = np.inf, None, None
    cur_lr = lr
    gen = torch.Generator().manual_seed(seed)
    for P in ladder:
        if prev is None:
            init = None
        else:
            if len(prev) > P: raise ValueError("ladder must be non-decreasing")
            init = list(prev) + [torch.randn(Nd, generator=gen, dtype=RDTYPE)*jitter
                                 for _ in range(P-len(prev))]
        _sync(device); t0 = time.time()
        model = MPLC(opt, m_list, P, init=init, gen=gen, device=device)
        v, mk = _train(model, epochs, cur_lr)
        _sync(device)
        if verbose:
            print(f"      {P:3d} planes -> Tr(F^-1) = {v:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if v < best_val: best_val, best_masks = v, mk
        prev = mk; cur_lr *= lr_decay
        if device.type == "cuda":
            del model; torch.cuda.empty_cache()
    return best_val, best_masks

def donn_detector(opt, masks, device=DEVICE):
    psi = torch.tensor(opt['psi'], dtype=torch.complex128)
    for mk in masks:
        m = mk.detach().to(device="cpu", dtype=torch.float64)
        psi = psi * torch.exp(1j*m.to(torch.complex128)).unsqueeze(0)
        psi = torch.fft.fftshift(torch.fft.fft(torch.fft.ifftshift(psi, dim=1),
                                               norm='ortho'), dim=1)
    return (torch.abs(psi)**2).numpy()

## Monte-Carlo validation

Poisson shot noise at $N = 10^5$ photons per trial, $T = 5000$ trials, matched (inverse-variance-weighted) linear estimator; returns the per-photon $\mathrm{Tr\,cov}$.

In [ ]:
def mc_pervar(w, dw, P, N=MC_NPH, T=MC_RUNS, seed=0):
    u0 = w @ P; D = dw @ P
    F = (D/np.sqrt(u0)) @ (D/np.sqrt(u0)).T; Finv = np.linalg.inv(F)
    rng = np.random.default_rng(seed)
    n = rng.poisson(N*u0, size=(T, len(u0)))
    est = ((n/u0) @ D.T) @ Finv.T / N
    cov = np.cov(est.T) if D.shape[0] > 1 else np.array([[est[:,0].var(ddof=1)]])
    return np.trace(cov) * N

## Sweep over $k_2$

Dense curves (DI, QCRB, ansatz bound, closed form) over all $m_2$ — cheap NumPy. Sparse points (NHCRB SDP, DONN training, Monte Carlo) — expensive; after every point the running results are checkpointed, so an interrupted run can still be plotted from `data/WZL_Fig2_ckpt.npz`.

In [ ]:
def sweep_two(opt, m1, m2_sparse, ladder=LADDER, epochs=EPOCHS,
              mc_T=MC_RUNS, mc_N=MC_NPH, do_nhcrb=True, ckpt=None, verbose=True):
    L = opt['L']
    m2_dense = [m for m in range(1, opt['M_max']-1) if m != m1]
    di_d, q_d, ub_d, cf_d = [], [], [], []
    for m2 in m2_dense:
        w, dw = cos_derivs([m1, m2], opt)
        re, dr, K = eigenbasis(w, [dw[0], dw[1]], opt)
        di_d.append(crb_di(w, [dw[0], dw[1]], opt))
        ub, trq = ansatz_ub(re, dr)
        q_d.append(trq); ub_d.append(ub)
        cf_d.append(analytic_ub(m1/(2*L), m2/(2*L), opt))
    if verbose:
        print(f"dense curves done ({len(m2_dense)} pts)", flush=True)

    # sparse (expensive): NHCRB SDP, DONN, Monte Carlo
    nh, dn, di_n, q_n, ub_n, mc_di, mc_dn, done = [], [], [], [], [], [], [], []
    if verbose:
        print(f"{'m2':>4} {'nu2/nuc':>8} {'DI':>10} {'DI_MC':>10} {'QCRB':>10} "
              f"{'NHCRB':>10} {'ansatz':>10} {'DONN':>10} {'DONN_MC':>10} {'t':>7}",
              flush=True)
    for m2 in m2_sparse:
        t0 = time.time()
        w, dw = cos_derivs([m1, m2], opt)
        re, dr, K = eigenbasis(w, [dw[0], dw[1]], opt)
        cdi = crb_di(w, [dw[0], dw[1]], opt)
        ub, cq = ansatz_ub(re, dr)
        cn = crb_nhcrb(re, dr) if do_nhcrb else np.nan
        bd, mkd = donn_planes(opt, [m1, m2], ladder=ladder, epochs=epochs, verbose=verbose)
        v_di = mc_pervar(w, dw, opt['P_wf'], N=mc_N, T=mc_T)
        v_dn = mc_pervar(w, dw, donn_detector(opt, mkd), N=mc_N, T=mc_T)
        nh.append(cn); dn.append(bd); di_n.append(cdi); q_n.append(cq); ub_n.append(ub)
        mc_di.append(v_di); mc_dn.append(v_dn); done.append(m2)
        if verbose:
            print(f"{m2:4d} {m2/(2*L)/opt['nu_coh']:8.3f} {cdi:10.3f} {v_di:10.3f} "
                  f"{cq:10.3f} {cn:10.3f} {ub:10.3f} {bd:10.3f} {v_dn:10.3f} "
                  f"{time.time()-t0:6.0f}s", flush=True)
        if ckpt:                                     # checkpoint after every point
            np.savez(ckpt, m1=m1, nu1=m1/(2*L), m2_sparse=np.array(done),
                     nu2_sparse=np.array([m/(2*L) for m in done]),
                     nhcrb=np.array(nh), donn=np.array(dn), di_num=np.array(di_n),
                     qcrb_num=np.array(q_n), ub_num=np.array(ub_n),
                     mc_di=np.array(mc_di), mc_donn=np.array(mc_dn), mc_T=mc_T,
                     m2_dense=np.array(m2_dense),
                     nu2_dense=np.array([m/(2*L) for m in m2_dense]),
                     di_dense=np.array(di_d), qcrb_dense=np.array(q_d),
                     ub_dense=np.array(ub_d), cf_dense=np.array(cf_d))

    return dict(m1=m1, nu1=m1/(2*L),
                m2_sparse=np.array(m2_sparse),
                nu2_sparse=np.array([m/(2*L) for m in m2_sparse]),
                nhcrb=np.array(nh), donn=np.array(dn), di_num=np.array(di_n),
                qcrb_num=np.array(q_n), ub_num=np.array(ub_n),
                mc_di=np.array(mc_di), mc_donn=np.array(mc_dn), mc_T=mc_T,
                m2_dense=np.array(m2_dense),
                nu2_dense=np.array([m/(2*L) for m in m2_dense]),
                di_dense=np.array(di_d), qcrb_dense=np.array(q_d),
                ub_dense=np.array(ub_d), cf_dense=np.array(cf_d))

## Figure

In [ ]:
# ---------------- cosmetic ----------------
FIGSIZE = (9, 6)
DPI = 300

FONT_SIZE = 15
LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14

QCRB_COLOR = "crimson"
ANSATZ_COLOR = "tab:blue"
DI_COLOR = "0.35"
DONN_COLOR = "darkorange"
NHCRB_COLOR = "tab:blue"
MC_COLOR = "black"
K1_COLOR = "darkgreen"

LINE_WIDTH = 2.2
MARKER_SIZE = 8

Y_SCALE = "log"    
LEGEND_LOCATION = "upper left"
LEGEND_COLUMNS = 1
SHOW_GRID = True
SHOW_CUTOFF_LABELS = True


def make_plot(opt, res, fname="figures/WZL_Fig2.png"):
    mpl.rcdefaults()
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "mathtext.fontset": "dejavusans",
        "font.size": FONT_SIZE,
        "axes.labelsize": LABEL_SIZE,
        "xtick.labelsize": TICK_SIZE,
        "ytick.labelsize": TICK_SIZE,
    })

    # ---------- axes data ----------
    xd = res["nu2_dense"] / opt["nu_coh"]
    xs = res["nu2_sparse"] / opt["nu_coh"]

    nu1 = float(res["nu1"])
    o1 = otf_triangle(nu1, opt)

    nu = np.linspace(1e-3, 1.97 * opt["nu_coh"], 400)
    o2 = otf_triangle(nu, opt)
    xn = nu / opt["nu_coh"]

    # ---------- plot ----------
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Nagaoka band: the true NHCRB lies between the QCRB and the SLD-ansatz bound
    ax.fill_between(xd, res["qcrb_dense"], res["ub_dense"],
                    color=ANSATZ_COLOR, alpha=0.14, linewidth=0)

    ax.plot(xd, res["qcrb_dense"], color=QCRB_COLOR, linewidth=LINE_WIDTH,
            label="QCRB (numerical)")
    ax.plot(xn, 2/o1 + 2/o2, color=QCRB_COLOR, linewidth=1.1, linestyle="--",
            alpha=0.55, label="QCRB (analytic ref.)")
    ax.plot(xd, res["di_dense"], color=DI_COLOR, linewidth=LINE_WIDTH,
            label="DI (numerical)")
    ax.plot(xn, 2/o1**2 + 2/o2**2, color=DI_COLOR, linewidth=1.1, linestyle="--",
            alpha=0.55, label="DI (analytic ref.)")
    ax.plot(xd, res["ub_dense"], color=ANSATZ_COLOR, linewidth=LINE_WIDTH,
            label="NHCRB (ansatz; upper bd.)")

    if np.isfinite(res["nhcrb"]).any():
        ax.plot(xs, res["nhcrb"], marker="s", linestyle=":", color=NHCRB_COLOR,
                markerfacecolor="none", markeredgewidth=1.6,
                markersize=MARKER_SIZE + 2, label="NHCRB (SDP)")
    ax.plot(xs, res["donn"], marker="D", linestyle="none", color=DONN_COLOR,
            markersize=MARKER_SIZE, label="DONN CRB")
    ax.plot(xs, res["mc_di"], marker="x", linestyle="none", color=MC_COLOR,
            markersize=MARKER_SIZE, markeredgewidth=1.5, label="DI (MC variance)")
    ax.plot(xs, res["mc_donn"], marker="+", linestyle="none", color=MC_COLOR,
            markersize=MARKER_SIZE + 3, markeredgewidth=1.5, label="DONN (MC variance)")
    ax.set_yscale(Y_SCALE)
    ax.set_xlim(0, max(xs) * 1.05)

    positive_values = np.concatenate([
        np.asarray(res["qcrb_dense"]),
        np.asarray(res["di_dense"]),
        np.asarray(res["ub_dense"]),
    ])
    positive_values = positive_values[np.isfinite(positive_values) & (positive_values > 0)]
    if Y_SCALE == "log":
        ax.set_ylim(positive_values.min() / 1.6, positive_values.max() * 3)

    y_top = ax.get_ylim()[1]
    y_bottom = ax.get_ylim()[0]

    if SHOW_CUTOFF_LABELS:
        for xc, text, ha, offset in [(1.0, "coherent\ncutoff", "left", 0.025),
                                     (2.0, "incoherent\ncutoff", "right", -0.025)]:
            ax.axvline(xc, color="0.35", linestyle=":", linewidth=1.2)
            ax.text(xc + offset,
                    y_top / 1.7 if Y_SCALE == "log" else y_top * 0.85,
                    text, horizontalalignment=ha, verticalalignment="top",
                    color="0.35", fontsize=TICK_SIZE,
                    bbox={"facecolor": "white", "edgecolor": "none",
                          "alpha": 0.8, "pad": 2})

    nu1_xc = nu1 / opt["nu_coh"]
    ax.axvline(nu1_xc, color=K1_COLOR, linestyle="--", linewidth=1.3)
    ax.text(nu1_xc + 0.025,
            y_bottom * 4.5 if Y_SCALE == "log" else y_bottom,
            r"$k_1$ fixed", horizontalalignment="left", verticalalignment="bottom",
            color=K1_COLOR, fontsize=TICK_SIZE,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 2})


    ax.set_xlabel(r"Normalized spatial frequency "
                  r"$\left(\frac{k_2}{2\pi\,\mathrm{NA}/\lambda}\right)$")
    ax.set_ylabel(r"Per-photon variance for $\{a_1,a_2\}$")
    ax.legend(fontsize=LEGEND_SIZE, loc=LEGEND_LOCATION, ncol=LEGEND_COLUMNS, frameon=True)
    if SHOW_GRID:
        ax.grid(True, which="both", linestyle="-.", alpha=0.25)
    ax.tick_params(which="both", direction="in", top=True, right=True)
    fig.tight_layout()

    out = Path(fname)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    fig.savefig(out.with_suffix(".svg"), bbox_inches="tight")
    print("saved", out, "and", out.with_suffix(".svg"), flush=True)
    plt.show()

## Run simulation

In [ ]:
M1        = 20        # fixed mode index k1  (nu1/nu_coh ~ 0.36)
M2_SPARSE = [4, 12, 28, 38, 48, 58, 68, 78, 86, 92, 97, 101, 104, 106, 108]
DO_NHCRB  = True      # False skips the SDP (memory-heavy)

FIG_DIR, DATA_DIR = Path("figures"), Path("data")
DATA_NPZ = DATA_DIR / "WZL_Fig2.npz"
CKPT_NPZ = None # write path here ("data/...") if there's a need to save datapoints as the code runs

opt = optics_exact_1d()
print(f"N={opt['N']} L={opt['L']} K={opt['K']} Nd={opt['Nd']} "
      f"M_max={opt['M_max']} capture={opt['capture'].mean():.4f}", flush=True)
print(f"SDP block side = (M+1)*K_trunc  (K_full={opt['K']}, "
      f"truncated at tol={SDP_TRUNC_TOL})", flush=True)

m2_sparse = [m for m in M2_SPARSE if m != M1 and 1 <= m <= opt['M_max']-2]
print(f"m1={M1}  m2 sweep ({len(m2_sparse)} pts): {m2_sparse}", flush=True)

### Full sweep — **[slow]**

One DONN ladder + one SDP + Monte Carlo per sweep point. Prints a running table, checkpoints after every point, saves `data/WZL_Fig2.npz`, verifies the bound hierarchy $c_Q \le c_{\rm NH} \le c_{\rm ansatz}$, and draws the figure.

In [ ]:
t0 = time.time()
DATA_DIR.mkdir(exist_ok=True)
res = sweep_two(opt, m1=M1, m2_sparse=m2_sparse, ladder=LADDER, epochs=EPOCHS,
                mc_T=MC_RUNS, mc_N=MC_NPH, do_nhcrb=DO_NHCRB, ckpt=CKPT_NPZ)
np.savez(DATA_NPZ, **res)
print(f"\nsaved {DATA_NPZ}   [total {time.time()-t0:.0f}s]", flush=True)

# sanity checks: bound hierarchy  c_Q <= c_NH <= c_ansatz
ok_q = bool(np.all(res['ub_dense'] >= res['qcrb_dense'] - 1e-9))
print(f"ansatz >= QCRB on the dense grid: {ok_q}", flush=True)
if np.isfinite(res['nhcrb']).any():
    f = np.isfinite(res['nhcrb'])
    print("QCRB <= NHCRB <= ansatz at sparse pts:",
          bool(np.all(res['nhcrb'][f] >= res['qcrb_num'][f]-1e-6)
               and np.all(res['ub_num'][f] >= res['nhcrb'][f]-1e-6)), flush=True)

make_plot(opt, res)

### Re-plot from saved data — **[fast]**

In [ ]:
# Fast path: regenerate the figure from the saved arrays
res_saved = dict(np.load(DATA_NPZ))
make_plot(opt, res_saved)